In [1]:
# Import foundational data manipulation and numerical computation libraries
import pandas as pd 
import numpy as np

***Loading Dataset***

In [2]:
# Load the raw monthly revenue dataset into a pandas DataFrame
revenue = pd.read_csv('monthly_revenue.csv')

# Display top and bottom rows to inspect structure and initial data types
revenue

,Date,Land Class,Land Category,State,County,FIPS Code,Offshore Region,Revenue Type,Mineral Lease Type,Commodity,Product,Revenue
0,01/01/2003,Federal,Onshore,Arkansas,Sebastian,5131.0,NaN,Other revenues,Oil & Gas,Oil & gas (pre-production),NaN,63655.97
1,01/01/2003,Federal,Onshore,Colorado,Moffat,8081.0,NaN,Other revenues,Oil & Gas,Oil & gas (pre-production),NaN,6823.00
2,01/01/2003,Federal,Onshore,Idaho,Caribou,16029.0,NaN,Rents,Phosphate,Phosphate,NaN,-1941.00
3,01/01/2003,Federal,Onshore,California,Kern,6029.0,NaN,Rents,Oil & Gas,Oil & gas (pre-production),NaN,1523.00
4,01/01/2003,Federal,Onshore,California,Kern,6029.0,NaN,Other revenues,Oil & Gas,Oil & gas (pre-production),NaN,3846.86
...,...,...,...,...,...,...,...,...,...,...,...,...
413549,08/01/2026,Federal,Onshore,Montana,Blaine,30005.0,NaN,Other revenues,Oil & Gas,Oil & gas (pre-production),NaN,14625.22
413550,08/01/2026,Federal,Onshore,Montana,Blaine,30005.0,NaN,Royalties,Oil & Gas,Oil,Oil,21747.83
413551,08/01/2026,Federal,Onshore,Montana,Blaine,30005.0,NaN,Royalties,Oil & Gas,Gas,Unprocessed Gas,5411.17
413552,08/01/2026,Federal,Onshore,Montana,Blaine,30005.0,NaN,Royalties,Oil & Gas,Gas,Pipeline Fuel/Loss,276.55


***Data Cleaning, Preprocessing and Normalization***

In [3]:
# Check baseline row and column counts prior to preprocessing
print(f'Size of the dataset before cleaning: {revenue.shape}')

Size of the dataset before cleaning: (413554, 12)


In [4]:
# Count missing values per column to identify missingness patterns
revenue.isnull().sum()

Date                       0
Land Class                 0
Land Category              0
State                  20303
County                 20303
FIPS Code              20303
Offshore Region       403323
Revenue Type               0
Mineral Lease Type       753
Commodity                  0
Product               122243
Revenue                    0
dtype: int64

In [5]:
# Convert Date string to datetime object for time-series operations
revenue['Date'] = pd.to_datetime(revenue['Date'], format='%m/%d/%Y')

# Define categorical text columns requiring normalization
text_col = ['Land Class', 'Land Category', 'State', 'County',
            'Revenue Type', 'Mineral Lease Type', 'Commodity', 'Product']

# Trim leading/trailing whitespace and convert empty/string 'nan' representations to true None values
for col in text_col:
    revenue[col] = revenue[col].astype(str).str.strip()
    revenue[col] = revenue[col].replace({'nan': None, 'None': None, '': None})

In [6]:
# Fill missing categorical fields with explicit domain labels
revenue['State'] = revenue['State'].fillna("Not Applicable")
revenue['County'] = revenue['County'].fillna("Not Applicable")
revenue['Mineral Lease Type'] = revenue['Mineral Lease Type'].fillna("Other / Unclassified")
revenue['Product'] = revenue['Product'].fillna('Unspecified')

In [7]:
# Verify that all target categorical columns no longer contain null values
revenue.isnull().sum()

Date                       0
Land Class                 0
Land Category              0
State                      0
County                     0
FIPS Code              20303
Offshore Region       403323
Revenue Type               0
Mineral Lease Type         0
Commodity                  0
Product                    0
Revenue                    0
dtype: int64

In [8]:
# Drop 'Offshore Region' due to high missingness (>97%) and 'FIPS Code' due to redundancy with State/County
revenue = revenue.drop(columns=['Offshore Region','FIPS Code'])

In [9]:
revenue.isnull().sum()

Date                  0
Land Class            0
Land Category         0
State                 0
County                0
Revenue Type          0
Mineral Lease Type    0
Commodity             0
Product               0
Revenue               0
dtype: int64

In [10]:
# Check for exact duplicate rows across all remaining columns
revenue.duplicated().sum()

np.int64(0)

In [11]:
# Print dataset dimensions post-cleaning to confirm shape consistency
print(f"Size of dataset after cleaning: {revenue.shape}")

Size of dataset after cleaning: (413554, 10)


***Adding time intelligence and classification field***

In [12]:
# Extract temporal components to facilitate seasonal and trend analysis
revenue['Year'] = revenue['Date'].dt.year
revenue['Quarter'] = revenue['Date'].dt.to_period('Q').astype(str)
revenue['Month_Name'] = revenue['Date'].dt.strftime('%b')
revenue['Month'] = revenue['Date'].dt.month

In [13]:
# Define energy source groupings for aggregated commodity analysis
fossil_fuels = ['Oil', 'Gas', 'Oil & gas (pre-production)', 'Natural gas liquids', 'Coal']
renewables = ['Geothermal']

# Categorize commodities into Fossil Fuels, Renewable Energy, or Minerals & Other
revenue['Commodity_Group'] = revenue['Commodity'].apply(
    lambda x: 'Fossil Fuels' if x in fossil_fuels 
    else ('Renewable Energy' if x in renewables else 'Minerals & Other')
)

In [14]:
# Inspect cleaned dataset sample
revenue

,Date,Land Class,Land Category,State,County,Revenue Type,Mineral Lease Type,Commodity,Product,Revenue,Year,Quarter,Month_Name,Month,Commodity_Group
0,2003-01-01,Federal,Onshore,Arkansas,Sebastian,Other revenues,Oil & Gas,Oil & gas (pre-production),Unspecified,63655.97,2003,2003Q1,Jan,1,Fossil Fuels
1,2003-01-01,Federal,Onshore,Colorado,Moffat,Other revenues,Oil & Gas,Oil & gas (pre-production),Unspecified,6823.00,2003,2003Q1,Jan,1,Fossil Fuels
2,2003-01-01,Federal,Onshore,Idaho,Caribou,Rents,Phosphate,Phosphate,Unspecified,-1941.00,2003,2003Q1,Jan,1,Minerals & Other
3,2003-01-01,Federal,Onshore,California,Kern,Rents,Oil & Gas,Oil & gas (pre-production),Unspecified,1523.00,2003,2003Q1,Jan,1,Fossil Fuels
4,2003-01-01,Federal,Onshore,California,Kern,Other revenues,Oil & Gas,Oil & gas (pre-production),Unspecified,3846.86,2003,2003Q1,Jan,1,Fossil Fuels
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413549,2026-08-01,Federal,Onshore,Montana,Blaine,Other revenues,Oil & Gas,Oil & gas (pre-production),Unspecified,14625.22,2026,2026Q3,Aug,8,Fossil Fuels
413550,2026-08-01,Federal,Onshore,Montana,Blaine,Royalties,Oil & Gas,Oil,Oil,21747.83,2026,2026Q3,Aug,8,Fossil Fuels
413551,2026-08-01,Federal,Onshore,Montana,Blaine,Royalties,Oil & Gas,Gas,Unprocessed Gas,5411.17,2026,2026Q3,Aug,8,Fossil Fuels
413552,2026-08-01,Federal,Onshore,Montana,Blaine,Royalties,Oil & Gas,Gas,Pipeline Fuel/Loss,276.55,2026,2026Q3,Aug,8,Fossil Fuels


In [15]:
# Export the cleaned, enriched dataset to a CSV file for downstream modeling or dashboarding
revenue.to_csv('cleaned_monthly_revenue.csv', index=False)